<a href="https://colab.research.google.com/github/voskanyanelen33-art/Real-Estate-Data-Pipeline/blob/main/work1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import requests
from datetime import datetime
import time
from bs4 import BeautifulSoup
import json
import csv
import shutil
import pandas as pd
import numpy as np
if os.path.exists('archive_data'):
  for root, dirs, files in os.walk('archive_data'):
    for file in files:
      if file.lower().endswith('csv') or file.lower().endswith('json'):
        os.rename(os.path.join(root, file),os.path.join(root, 'backup_'+(str(datetime.now().strftime("%Y%m%d_%H%M%S"))+file)))
if  os.path.exists('archive_data'):
  for root, dirs, files in os.walk('archive_data'):
    for file in files:
      if (file.lower().endswith('csv') or file.lower().endswith('json')) and  'backup' in file:
        os.makedirs(os.path.join('archive_data', 'backup'), exist_ok=True)
        shutil.move(os.path.join(root, file),os.path.join('archive_data', 'backup', file))
else:
  os.makedirs('archive_data')
h1 = 'Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36 [ip:151.36.14.249]'
h2 = 'AppleCoreMedia/1.0.0.19H394 (iPhone; U; CPU OS 15_8_5 like Mac OS X; ar)'
h3 = 'Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Mobile Safari/537.36 [ip:151.47.199.163]'
h4 = 'Mozilla/5.0 (Linux; Android 15; SM-S918B Build/AP3A.240905.015.A2; wv) AppleWebKit/537.36 (KHTML, like Gecko) Version/4.0 Chrome/140.0.7339.155 Mobile Safari/537.36 Vinebre'
h5 = 'Mozilla/5.0 (Linux; arm_64; Android 12; SHARK KTUS-H0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.7204.105 YaBrowser/25.8.5.105.00 SA/3 Mobile Safari/537.36'
headers=[h1, h2, h3, h4, h5]
url = 'https://quotes.toscrape.com/'
current_headers = {'User-Agent': random.choice(headers)}
response = None

for attempt in range(1, 4):
    try:
        print(f"[INFO] Attempt {attempt}: Fetching data...")
        response = requests.get(url, headers=current_headers, timeout=5)
        response.raise_for_status()
        print("[SUCCESS] Data fetched successfully!")
        break

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"[WARNING] Attempt {attempt} failed: {e}")
        if attempt < 3:
            delay = 2 ** attempt
            print(f"[INFO] Waiting {delay} seconds before retry...")
            time.sleep(delay)
        else:
            print("[CRITICAL] All 3 attempts failed. Switching to emergency mode...")
            response = None
soup = BeautifulSoup(response.text, 'html.parser')
furniture_lst = []
for furn in soup.find_all('div', class_='quote'):
  furniture_dct = {}
  furniture_dct['Furn_descript'] = furn.find('span', class_ = 'text').text
  furniture_dct['Furn_name'] = furn.find('small', class_ = 'author').text
  furniture_dct['Furn_amenities'] =' | '.join( furn.find('div', class_ = 'tags').find('meta', class_ = 'keywords')['content'].split(','))
  furniture_lst.append(furniture_dct)
print(furniture_lst)

class ListingProcessor:

  def make_df(self,furniture_lst):
    df = pd.DataFrame(furniture_lst)
    df1 = df
    pd.set_option('display.max_colwidth', 100)
    mask = df1["Furn_descript"].str.len() >65

    df1["Furn_descript"] = df1["Furn_descript"].str[:65].str.strip()
    df1.loc[mask, "Furn_descript"] += "...Read More"
    len_df = pd.DataFrame(list(i for i in (((df['Furn_descript'].str.len())))))
    a = (np.array(df['Furn_name'].str[:4].str.upper()))
    b = ((list(i for i in len_df.astype(str)[0])))
    counter = np.arange(0, 10)
    count_df = pd.DataFrame(counter)
    counter = (np.array(list(i for i in count_df.astype(str)[0])))
    ID_lst = np.array(a) + np.array(b)+'_' + counter
    df1['Unique_id'] = ID_lst
    self.df1 = df1
    return ID_lst

  def make_files(self):
    self.df1.to_csv("archive_data/listings.csv", index = False, encoding = "utf-8-sig")
    self.df1.to_json("archive_data/listings.json", orient = "records", indent = 2, force_ascii = False)


object=ListingProcessor()
object.make_df(furniture_lst)
object.make_files()

